In [5]:
# %%writefile ../data/download_data.py
import os
import zipfile
import kagglehub  # pip install kagglehub

def download_data(dataset_slug: str = "iamsouravbanerjee/animal-image-dataset-90-different-animals",
                   dest_dir: str = "data"):
    
    'we use this function for downloading our dara from kagglehub'
    os.makedirs(dest_dir, exist_ok=True)

    # starting the downlaoding
    cache_path = kagglehub.dataset_download(dataset_slug)
    print(f"Downloaded to the path : {cache_path}")

    # extracting the file
    for f in os.listdir(cache_path):
        if f.endswith(".zip"):
            with zipfile.ZipFile(os.path.join(cache_path, f), "r") as z:
                z.extractall(dest_dir)
    else:
        if not os.listdir(dest_dir):
            os.symlink(cache_path, dest_dir + "_link")

    return dest_dir


In [7]:
download_data()

Resuming download from 362807296 bytes (325107375 bytes left)...
Resuming download to C:\Users\sadegh\.cache\kagglehub\datasets\iamsouravbanerjee\animal-image-dataset-90-different-animals\5.archive (362807296/687914671) bytes left.


100%|██████████| 656M/656M [00:42<00:00, 7.60MB/s]

Extracting files...


Downloaded to the path : C:\Users\sadegh\.cache\kagglehub\datasets\iamsouravbanerjee\animal-image-dataset-90-different-animals\versions\5


'data'

In [1]:
# %%writefile src/model.py

import torch
import torchvision
from torch import nn
def create_effnetB2_model(num_of_classes = 90 , 
                          seed = 42):
  

  # First we download our model EfficientNet_B2_Weights
  weights = torchvision.models.EfficientNet_B2_Weights.DEFAULT
  transform = weights.transforms()
  model = torchvision.models.efficientnet_b2(weights=weights)

  # 4. Freeze all layers in base model
  for param in model.parameters():
    param.requires_grad = False

  torch.manual_seed(42)
  torch.cuda.manual_seed(42)
  model.classifier = torch.nn.Sequential(
      nn.Dropout(p=0.3, inplace=True),
      nn.Linear(in_features=1408, out_features=num_of_classes),
  )

  return model , transform

In [2]:
# !pip install torchinfo

from torchinfo import summary
from src.model import create_effnetB2_model
effnet_b2_model , effnet_b2_transform= create_effnetB2_model()

# Print EffNetB2 model summary (uncomment for full output) 
summary(effnet_b2_model, 
        input_size=(1, 3, 224, 224),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
EfficientNet (EfficientNet)                                  [1, 3, 224, 224]     [1, 90]              --                   Partial
├─Sequential (features)                                      [1, 3, 224, 224]     [1, 1408, 7, 7]      --                   False
│    └─Conv2dNormActivation (0)                              [1, 3, 224, 224]     [1, 32, 112, 112]    --                   False
│    │    └─Conv2d (0)                                       [1, 3, 224, 224]     [1, 32, 112, 112]    (864)                False
│    │    └─BatchNorm2d (1)                                  [1, 32, 112, 112]    [1, 32, 112, 112]    (64)                 False
│    │    └─SiLU (2)                                         [1, 32, 112, 112]    [1, 32, 112, 112]    --                   --
│    └─Sequential (1)                                        [1, 32, 112, 112]    [1, 1

In [9]:
# %%writefile src/data_setup.py

import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

num_of_workers = os.cpu_count()

# we use this function to make our test and train Dataloaders

def create_dataloader(transform : int , 
                      batch_size: int , 
                      train_dir:str ,
                      num_of_workers = num_of_workers):
  
  train_data =  datasets.ImageFolder(train_dir,transform=transform)

  classes_name = train_data.classes 

  train_Dataloder = DataLoader(
      train_data ,
      batch_size = batch_size , 
      shuffle = True , 
      num_workers = num_of_workers
  )

  return train_Dataloder , classes_name
  
  

Overwriting src/data_setup.py


In [1]:
# %%writefile src/train.py

import torch
from torch import nn

from src.engine import train
from src.data_setup import create_dataloader
from src.model import create_effnetB2_model

#import the model and transform function 
effnet_b2_model , effnet_b2_transform= create_effnetB2_model()

train_dir = 'data/animals/train'

device = 'cuda' if torch.cuda.is_available() else 'cpu' #set the device

#set the optimizer
optimizer = torch.optim.Adam(params=effnet_b2_model.parameters(),
                             lr=1e-3)
# Setup loss function
loss_fn = torch.nn.CrossEntropyLoss()

#first we get our train Dataloader and classes
train_Dataloader , classes_name = create_dataloader(transform=effnet_b2_transform , train_dir=train_dir , batch_size=32)

effnetb2_results = train(model=effnet_b2_model,
                                train_dataloader=train_Dataloader,
                                epochs=10,
                                optimizer=optimizer,
                                loss_fn=loss_fn,
                                device=device)

  0%|          | 0/10 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x0000017414828680>
Traceback (most recent call last):
  File "C:\Users\sadegh\pytorch_gpu\env2\Lib\site-packages\torch\utils\data\dataloader.py", line 1686, in __del__
    self._shutdown_workers()
  File "C:\Users\sadegh\pytorch_gpu\env2\Lib\site-packages\torch\utils\data\dataloader.py", line 1650, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "C:\Users\sadegh\AppData\Local\Programs\Python\Python313\Lib\multiprocessing\process.py", line 149, in join
    res = self._popen.wait(timeout)
  File "C:\Users\sadegh\AppData\Local\Programs\Python\Python313\Lib\multiprocessing\popen_spawn_win32.py", line 114, in wait
    res = _winapi.WaitForSingleObject(int(self._handle), msecs)
KeyboardInterrupt: 


KeyboardInterrupt: 

In [23]:
%%writefile src/utils.py

import torch
import os 
import matplotlib.pyplot as plt


def save_model(model):
    if not os.path.exists('../models'):
        os.makedirs('../models')
        print('models folder created')
        
    torch.save(model.state_dict() , '../models/model.pth')
    print("✅ Model Saved")

def plotresults(train_accs , train_losses):
    plt.figure(figsize=(12,5))

    plt.subplot(1,2,1)
    plt.plot(train_losses, label='Train Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training Loss')
    plt.legend()

    plt.subplot(1,2,2)
    plt.plot(train_accs, label='Train Accuracy', color='orange')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.title('Training Accuracy')
    plt.legend()

    plt.tight_layout()
    plt.show()

Writing ../src/utils.py


In [ ]:
save_model(effnet_b2_model)

In [ ]:
plotresults(train_accs=effnetb2_results["train_acc"] , train_losses=effnetb2_results["train_loss"])

In [37]:
# %%writefile src/predict.py

import torchvision
import torch
from timeit import default_timer as timer
from src.model import create_effnetB2_model
from src.data_setup import create_dataloader
# we set the device
device = 'cuda' if torch.cuda.is_available() else 'cpu'

test_dir = 'data/animals/test'
test_data , class_names = create_dataloader(train_dir=test_dir , batch_size=32 , transform=transform)

print(class_names)
# import the model and model transform
model , transform = create_effnetB2_model()
model.load_state_dict(torch.load('models/model.pth'))


def predict(img , topk = 3):

    # starting the timer
    start_time = timer()

    # unsqueeze the image to fit the model 
    img = transform(img).unsqueeze(0).to(device)

    model.eval()
    model.to(device)
    with torch.inference_mode():

        pred_probs = torch.softmax(model(img) , dim=1)
        
        # We use topk to have 3 with the most probability
        top_probs, top_idxs = torch.topk(pred_probs, k=topk, dim=1)
        
    # make a dict of label probability
    pred_labels_and_probs = {
        class_names[idx]: float(prob)
        for idx, prob in zip(top_idxs[0], top_probs[0])
    }


    pred_time = round(timer() - start_time , 5)

    return pred_labels_and_probs , pred_time

['antelope', 'badger', 'bat', 'bear', 'bee', 'beetle', 'bison', 'boar', 'butterfly', 'cat', 'caterpillar', 'chimpanzee', 'cockroach', 'cow', 'coyote', 'crab', 'crow', 'deer', 'dog', 'dolphin', 'donkey', 'dragonfly', 'duck', 'eagle', 'elephant', 'flamingo', 'fly', 'fox', 'goat', 'goldfish', 'goose', 'gorilla', 'grasshopper', 'hamster', 'hare', 'hedgehog', 'hippopotamus', 'hornbill', 'horse', 'hummingbird', 'hyena', 'jellyfish', 'kangaroo', 'koala', 'ladybugs', 'leopard', 'lion', 'lizard', 'lobster', 'mosquito', 'moth', 'mouse', 'octopus', 'okapi', 'orangutan', 'otter', 'owl', 'ox', 'oyster', 'panda', 'parrot', 'pelecaniformes', 'penguin', 'pig', 'pigeon', 'porcupine', 'possum', 'raccoon', 'rat', 'reindeer', 'rhinoceros', 'sandpiper', 'seahorse', 'seal', 'shark', 'sheep', 'snake', 'sparrow', 'squid', 'squirrel', 'starfish', 'swan', 'tiger', 'turkey', 'turtle', 'whale', 'wolf', 'wombat', 'woodpecker', 'zebra']


In [29]:
import random
import torch
from PIL import Image
from pathlib import Path

from src.model import create_effnetB2_model



model , transform = create_effnetB2_model()
model.load_state_dict(torch.load('models/model.pth'))

test_dir = 'data/animals/test'




# Get a list of all test image filepaths
test_data_path = list(Path(test_dir).glob('*/*.jpg'))

# Randomly select a test image path
random_image_path = random.sample(test_data_path , k=1)[0]


# Open the target image
image = Image.open(random_image_path)
print(f"[INFO] Predicting on image at path: {random_image_path}\n")

# Predict on the target image and print out the outputs
pred_dict, pred_time = predict(img=image )
print(f"Prediction label and probability dictionary: \n{pred_dict}")
print(f"Prediction time: {pred_time} seconds")

[INFO] Predicting on image at path: data\animals\test\tiger\4a60c1dab1.jpg

Prediction label and probability dictionary: 
{'tiger': 0.9924594759941101, 'cat': 0.0026569280307739973, 'leopard': 0.0024484542664140463}
Prediction time: 0.17183 seconds


In [10]:
example_list = [[str(filepath)] for filepath in random.sample(test_data_path, k=3)]
example_list

[['data\\animals\\test\\seahorse\\26cd18c9c8.jpg'],
 ['data\\animals\\test\\porcupine\\80da7a6911.jpg'],
 ['data\\animals\\test\\butterfly\\26c0074cef.jpg']]

In [32]:
import gradio as gr 

title = "AnimalVision-90 🐈🐶🐄"
description = 'A deep learning model for identifying 90 different animal species from images with high accuracy.'

demo = gr.Interface(
    fn = predict,
    inputs = gr.Image(type="pil") ,
    outputs=[gr.Label(num_top_classes=3, label="Predictions"),
            gr.Number(label="Prediction time (s)")] ,
    examples=example_list, 
    title=title,
    description=description
)

demo.launch(debug=False, # print errors locally?
            share=True)

* Running on local URL:  http://127.0.0.1:7862

Could not create share link. Missing file: C:\Users\sadegh\.cache\huggingface\gradio\frpc\frpc_windows_amd64_v0.3. 

Please check your internet connection. This can happen if your antivirus software blocks the download of this file. You can install manually by following these steps: 

1. Download this file: https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_windows_amd64.exe
2. Rename the downloaded file to: frpc_windows_amd64_v0.3
3. Move the file to this location: C:\Users\sadegh\.cache\huggingface\gradio\frpc


C:\Users\sadegh\pytorch_gpu\env2\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
C:\Users\sadegh\pytorch_gpu\env2\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
C:\Users\sadegh\pytorch_gpu\env2\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
C:\Users\sadegh\pytorch_gpu\env2\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, req

In [36]:
import os 

if not os.path.exists('huggingface'):
    os.makedirs('huggingface')
    print('huggingface folder has been created')
else :
    print('you already have the folder')
    

you already have the folder


In [47]:
%%writefile huggingface/app.py
# the we should put model.py , model.pth in this folder
# then we starting to duild our app.py file to push it on hugging face

import torch
import os
from timeit import default_timer as timer
from model import create_effnetB2_model
from pathlib import Path

model , transform = create_effnetB2_model()
model.load_state_dict(torch.load('model.pth' , map_location='cpu'))

device = 'cuda' if torch.cuda.is_available() else 'cpu'

class_names = ['antelope', 'badger', 'bat', 'bear', 'bee', 'beetle',
               'bison', 'boar', 'butterfly', 'cat', 'caterpillar', 'chimpanzee',
               'cockroach', 'cow', 'coyote', 'crab', 'crow', 'deer', 'dog', 'dolphin', 
               'donkey', 'dragonfly', 'duck', 'eagle', 'elephant', 'flamingo', 'fly', 'fox',
               'goat', 'goldfish', 'goose', 'gorilla', 'grasshopper', 'hamster', 'hare', 'hedgehog',
               'hippopotamus', 'hornbill', 'horse', 'hummingbird', 'hyena', 'jellyfish', 'kangaroo', 'koala',
               'ladybugs', 'leopard', 'lion', 'lizard', 'lobster', 'mosquito', 'moth', 'mouse', 'octopus', 'okapi', 'orangutan',
               'otter', 'owl', 'ox', 'oyster', 'panda', 'parrot', 'pelecaniformes', 'penguin', 'pig', 'pigeon', 'porcupine', 'possum',
               'raccoon', 'rat', 'reindeer', 'rhinoceros', 'sandpiper', 'seahorse', 'seal', 'shark', 'sheep', 'snake', 'sparrow', 'squid', 'squirrel',
               'starfish', 'swan', 'tiger', 'turkey', 'turtle', 'whale', 'wolf', 'wombat', 'woodpecker', 'zebra']

def predict(img , topk = 3):

    # starting the timer
    start_time = timer()

    # unsqueeze the image to fit the model 
    img = transform(img).unsqueeze(0).to(device)

    model.eval()
    model.to(device)
    with torch.inference_mode():

        pred_probs = torch.softmax(model(img) , dim=1)
        
        # We use topk to have 3 with the most probability
        top_probs, top_idxs = torch.topk(pred_probs, k=topk, dim=1)
        
    # make a dict of label probability
    pred_labels_and_probs = {
        class_names[idx]: float(prob)
        for idx, prob in zip(top_idxs[0], top_probs[0])
    }


    pred_time = round(timer() - start_time , 5)

    return pred_labels_and_probs , pred_time
    
title = "AnimalVision-90 🐈🐶🐄"
description = 'A deep learning model for identifying 90 different animal species from images with high accuracy.'

example_list = list(Path('example').glob('*.jpg'))

demo = gr.Interface(
    fn = predict,
    inputs = gr.Image(type="pil") ,
    outputs=[gr.Label(num_top_classes=3, label="Predictions"),
            gr.Number(label="Prediction time (s)")] ,
    examples=example_list, 
    title=title,
    description=description
)

demo.launch(debug=False, # print errors locally?
            share=True)    

Overwriting huggingface/app.py


[WindowsPath('huggingface/example/0b30d5c395.jpg'),
 WindowsPath('huggingface/example/0cf04d0dab.jpg'),
 WindowsPath('huggingface/example/1a89f88226.jpg')]